In [1]:
import pandas as pd

In [2]:
import numpy as np

In [4]:
df = pd.read_csv("/content/powerplant_data (1).csv")

In [6]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [7]:
#AT = TEMP
#V = VACCUM
#AP = PRESSURE
#RH = HUMIDITY

#PE = PRODUCED ENERGY

In [8]:
df.isnull().sum()

,0
AT,0
V,0
AP,0
RH,0
PE,0


In [9]:
x = df.drop("PE", axis = 1)
y = df["PE"]

In [10]:
y.head()

,PE
0,480.48
1,445.75
2,438.76
3,453.09
4,464.43


In [11]:
#split data
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [12]:
x_train

,AT,V,AP,RH
5487,25.24,63.47,1011.30,66.21
3522,26.09,70.40,1007.41,85.37
6916,26.63,73.68,1015.15,85.13
7544,32.06,71.85,1007.90,56.44
7600,28.70,71.64,1007.11,69.85
...,...,...,...,...
5734,26.25,61.02,1011.47,71.22
5191,29.17,64.79,1016.43,61.05
5390,18.00,43.70,1015.40,61.28
860,26.73,68.84,1010.75,66.83


In [13]:
df.shape

(9568, 5)

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [15]:
x_test_scaled

array([[ 1.34499288,  0.23869298, -1.28658067, -1.10532538],
       [ 0.81095912,  1.36269098, -0.74140656,  0.26485915],
       [-0.2437241 , -0.73900436,  1.99970178, -0.19713193],
       ...,
       [-0.67068342, -1.15902881, -0.29951077, -0.10651852],
       [ 1.31420898,  1.33752097, -0.87346737, -0.44288647],
       [-0.2611237 , -0.27021304,  0.37433797,  1.10646548]])

In [16]:
import torch
import torch.nn as nn

x_train_tensor = torch.tensor(x_train_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32).view(-1, 1)

x_test_tensor = torch.tensor(x_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32).view(-1,1)



In [17]:
type(x_train_scaled)

numpy.ndarray

In [18]:
type(x_train_tensor)

torch.Tensor

In [20]:
type(y_train)
y_train.shape


(7654,)

In [21]:
 from torch.utils.data import TensorDataset, DataLoader

 train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
 test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

In [22]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32)

In [23]:
###Deep learning

In [24]:
# Define our ANN model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(x_train.shape[1], 6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output layer
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

In [25]:
import torch.optim as optim
model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [26]:
#train the ANN
train_losses = []
val_losses = []

best_val_loss = float('inf')
epochs = 100

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for xb, yb in train_loader:
    #xb = features of one batch
    #yb = lables of one batch
    optimizer.zero_grad()

    outputs = model(xb) #forward prop.....predicted outputs for this batch
    loss = criterion(outputs, yb) #compute loss
    loss.backward() #back prop.. compute gradients
    optimizer.step() #update parameters

    running_loss += loss.item() #loss is a tensor => py float

epoch_train_loss = running_loss / len(train_loader)
train_losses.append(epoch_train_loss)


#validation
model.eval()
with torch.no_grad():
  running_val_loss = 0.0
for xb, yb in test_loader:
  outputs = model(xb)
  loss = criterion(outputs, yb)
  running_val_loss += loss.item()

epoch_val_loss = running_val_loss / len(test_loader)
val_losses.append(epoch_val_loss)

print(f"epoch {epoch+1}/{epoch} ===> train loss = ${epoch_train_loss} & val loss = ${epoch_val_loss}")

if epoch_val_loss < best_val_loss:
  best_val_loss = epoch_val_loss
  torch.save(model.state_dict(), "best_model.pth")


epoch 100/99 ===> train loss = $21.265660132964452 & val loss = $20.10149574279785


In [27]:
#loading the best model
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [29]:
#Evaluate our model
model.eval()
with torch.no_grad():
  train_preds = model(x_train_tensor)
  test_preds = model(x_test_tensor)

  train_mse_loss = criterion(train_preds, y_train_tensor)
  test_mse_loss = criterion(test_preds, y_test_tensor)

print("Trainig MSE: ", train_mse_loss.item())
print("Testing MSE: ", test_mse_loss.item())

Trainig MSE:  21.58396339416504
Testing MSE:  20.110427856445312


In [30]:
from sklearn.metrics import r2_score
print ("r2 score: ", r2_score(y_test, test_preds))

r2 score:  0.9297192436147712


In [32]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns = ["Predicted Values"])
actual_df = pd.DataFrame(y_test.values, columns = ["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)

,Predicted Values,Actual Values
0,436.402252,433.27
1,437.904816,438.16
2,459.765930,458.42
3,474.387299,480.82
4,435.295441,441.41
...,...,...
1909,451.709686,456.70
1910,432.735748,438.04
1911,467.606720,467.80
1912,432.170074,437.14
